In [2]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import numpy as np
import json
from captum.attr import IntegratedGradients


In [3]:
SEED = 4266
DATA_DIR = "../../data/processed"
# DROP_OUT = 0.3
# LEARNING_RATE = 1e-3
# HIDDEN_DIM = 100
INPUT_DIM = 59
OUTPUT_DIM = 5
# LAYER_DIM = 1
EPOCHS = 20
PATIENCE = 4
MIN_DELTA = 1e-4

N_STEPS = 100

In [4]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [5]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim  = layer_dim
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim,
                            batch_first=True,
                            dropout=dropout if layer_dim > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.attn    = nn.Linear(hidden_dim, 1)
        self.fc      = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        #we reset the hidden state for each batch of sequences so that they are independent of each other. This is important for training on batches of sequences.
        out, _ = self.lstm(x)                       # (B, T, H) — zeros state by default
        # attention pooling over time
        scores  = self.attn(out).squeeze(-1)         # (B, T)
        #weight of each timestep
        weights = torch.softmax(scores, dim=1)       # (B, T)
        #weighted average
        pooled  = (out * weights.unsqueeze(-1)).sum(1)  # (B, H)
        #regularisation
        pooled  = self.dropout(pooled)
        return self.fc(pooled)


def load_baseline():
    hidden_dim = 128
    layer_dim = 2
    dropout = 0.2
    lr = 1e-3
    weight_decay = 0
    epochs = 100
    seed = SEED
    patience = 10
    optimizer_name = "adam"
    batch_size = 128
    min_delta = 0

    #set seed for reproducibility
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # np.random.seed(seed)


    #create LSTM
    model = LSTMModel(INPUT_DIM, hidden_dim, layer_dim, OUTPUT_DIM, dropout).to(device)
    model.load_state_dict(torch.load("lstm_model.pt", map_location=device))##, weights_only=True))
    #disable dropout
    model.eval()
    return model




In [6]:
model = load_baseline()


In [7]:

torch.backends.cudnn.enabled = False 

# feats
X_test_raw  = np.load(f"{DATA_DIR}/X_test.npy")


# labels
y_test_raw  = np.load(f"{DATA_DIR}/y_test.npy")

X_test = torch.tensor(X_test_raw, dtype=torch.float32).to(device)
Y_test = torch.tensor(y_test_raw, dtype=torch.long)
y_test_t = torch.tensor(np.argmax(y_test_raw, axis=1), dtype=torch.long).to(device)

#Captum requires gradients to be tracked


In [8]:
def ig_attributions_lstm(model, X_test, y_test, idx, n_steps=N_STEPS,
                         chunk=64, return_delta=False):

    model.eval()
    ig = IntegratedGradients(model)

    attr_list, y_list, pred_list, delta_list = [], [], [], []

    for i in range(0, len(idx), chunk):
        j = idx[i:i + chunk]
        x = X_test[j].to(device).requires_grad_()   # (B, T, F) — real samples
        y = y_test[j].to(device)

        with torch.no_grad():
            pred = model(x).argmax(dim=1)           # (B,)

        out = ig.attribute(
            inputs=x,
            baselines=torch.zeros_like(x),
            target=pred,
            n_steps=n_steps,
            internal_batch_size=256,
            return_convergence_delta=return_delta,
        )
        if return_delta:
            a_x, delta = out
            delta_list.append(delta.detach().cpu())
        else:
            a_x = out

        attr_list.append(a_x.detach().cpu())
        y_list.append(y.cpu())
        pred_list.append(pred.cpu())

    attr        = torch.cat(attr_list,  dim=0)      # (N, T, F)
    true_labels = torch.cat(y_list,     dim=0)      # (N,)
    pred_labels = torch.cat(pred_list,  dim=0)      # (N,)

    if return_delta:
        return attr, true_labels, pred_labels, torch.cat(delta_list, dim=0)
    return attr, true_labels, pred_labels

In [9]:
def balanced_idx(y, per_class=200, seed=42):
    rng = np.random.default_rng(seed)
    idx = [rng.choice(np.where(y == c)[0], min(per_class, (y == c).sum()), replace=False)
           for c in np.unique(y)]
    return torch.tensor(np.concatenate(idx))

idx = balanced_idx(y_test_t.numpy(), per_class=200)
attr, y_true, y_pred, delta = ig_attributions_lstm(model, X_test, y_test=y_test_t, idx=idx, n_steps=N_STEPS, chunk=64, return_delta=True)

In [10]:
def calculate_importance_lstm(attr):
    abs_attr  = attr.abs()
    feat_imp  = abs_attr.sum(dim=1)        # collapse time
    time_imp  = abs_attr.sum(dim=2)        # collapse features
    total_imp = abs_attr.sum(dim=(1, 2))   # collapse both
    return feat_imp, time_imp, total_imp


feat_importance, time_importance, total_importance = calculate_importance_lstm(attr)

In [11]:
print(f"feat_importance: {feat_importance.shape}")
print(f"time_importance: {time_importance.shape}")
print(f"total_importance: {total_importance.shape}")

feat_importance: torch.Size([412, 59])
time_importance: torch.Size([412, 5])
total_importance: torch.Size([412])


In [12]:
with open("../../data/processed/feature_cols.json") as f:
    feature_info = json.load(f)

feature_names = (feature_info["numeric"]+ feature_info["categorical"])


In [13]:
y_true = y_true.detach().cpu()
y_pred = y_pred.detach().cpu()


In [14]:
feat_df = (
    pd.DataFrame({
        "feature":            feature_names,                                   # 59
        "feature_importance": feat_importance.detach().mean(0).cpu().numpy(),  # 59
    })
    .sort_values("feature_importance", ascending=False)
    .reset_index(drop=True)
)
feat_df["share"] = feat_df["feature_importance"] / feat_df["feature_importance"].sum()
feat_df["rank"]  = feat_df["feature_importance"].rank(ascending=False).astype(int)
display(feat_df.head(20))

,feature,feature_importance,share,rank
0,mean,1.511050,0.095353,1
1,src_ttl_mean,1.448455,0.091403,2
2,src_jitter_mean,1.142060,0.072068,3
3,run_time_sum,1.141944,0.072061,4
4,src_loss,1.016468,0.064143,5
5,num_dst_ports,0.983352,0.062053,6
6,dst_ttl_mean,0.893338,0.056373,7
7,src_int_pkt_mean,0.869366,0.054860,8
8,src_load,0.768584,0.048501,9
9,tot_rate,0.592773,0.037406,10


In [15]:
time_df = (
    pd.DataFrame({
        "timestep":       [f"t{i}" for i in range(time_importance.shape[1])],  # 5
        "time_importance": time_importance.detach().mean(0).cpu().numpy(),      # 5
    })
    .sort_values("time_importance", ascending=False)
    .reset_index(drop=True)
)
time_df["share"] = time_df["time_importance"] / time_df["time_importance"].sum()
display(time_df)

,timestep,time_importance,share
0,t0,4.356922,0.274939
1,t1,3.690536,0.232887
2,t2,3.148811,0.198702
3,t3,2.490661,0.157170
4,t4,2.159965,0.136302


In [16]:
sample_df = pd.DataFrame({
    "y_true":           y_true.detach().cpu().numpy(),
    "y_pred":           y_pred.detach().cpu().numpy(),
    "total_importance": total_importance.detach().cpu().numpy(),   # 1 per sample
})
sample_df["correct"] = sample_df["y_true"] == sample_df["y_pred"]

print("total attribution per sample — summary")
print(sample_df["total_importance"].describe())

print(sample_df.groupby("y_true")["total_importance"].mean())

total attribution per sample — summary
count    412.000000
mean      15.846893
std       14.521371
min        2.066148
25%        3.149951
50%       10.182894
75%       25.392333
max       51.405720
Name: total_importance, dtype: float64
y_true
0    12.058900
1    18.997412
2    38.226427
3    26.987798
4     3.872642
Name: total_importance, dtype: float64


In [17]:
class_names = ["Backdoor", "CommInj", "DoS", "Reconn", "normal"]


tp = y_true == y_pred
per_class_results = {}

# per attack feature importance
# focusing on correctly predicted classes
for c, name in enumerate(class_names):

    mask = tp & (y_true == c)

    if mask.sum().item() == 0:
        continue

    scores = feat_importance[mask].mean(dim=0)

    per_attack = pd.DataFrame({
        "feature": feature_names,
        "importance": scores.numpy()
    })

    per_attack = per_attack.sort_values("importance",ascending=False)
    per_class_results[name] = per_attack

    print(f"\n{name}: {mask.sum().item()} samples")
    display(per_attack.head(10))


Backdoor: 29 samples


,feature,importance
26,dst_ttl_mean,2.210439
7,dst_bytes,2.117897
28,dst_app_bytes,2.058988
10,dst_load,0.883191
17,loss,0.835212
35,dst_int_pkt_mean,0.662209
16,dst_loss,0.642926
4,dst_pkts,0.435602
25,src_ttl_mean,0.364012
32,src_jitter_mean,0.315080



CommInj: 49 samples


,feature,importance
15,src_loss,7.878010
17,loss,2.563366
4,dst_pkts,1.175667
13,dst_rate,1.124857
10,dst_load,1.122708
27,src_app_bytes,0.727025
26,dst_ttl_mean,0.711846
56,dport_modbus,0.574396
16,dst_loss,0.497609
23,idle_mean,0.356246



DoS: 91 samples


,feature,importance
0,mean,6.716015
31,run_time_sum,4.973713
25,src_ttl_mean,4.952593
34,src_int_pkt_mean,3.218233
32,src_jitter_mean,3.102820
9,src_load,3.000921
14,tot_rate,2.512510
54,dport_http,2.295187
11,tot_load,1.382186
12,src_rate,1.328623



Reconn: 29 samples


,feature,importance
2,num_dst_ports,11.869364
26,dst_ttl_mean,5.434591
49,src_port_dynamic,1.528785
25,src_ttl_mean,1.452119
39,proto_6,1.409003
58,dport_registered,1.068568
9,src_load,1.031280
23,idle_mean,0.746685
52,dport_dynamic,0.723887
50,src_port_registered,0.545510



normal: 200 samples


,feature,importance
32,src_jitter_mean,0.753580
26,dst_ttl_mean,0.471868
25,src_ttl_mean,0.384255
23,idle_mean,0.371404
34,src_int_pkt_mean,0.251678
27,src_app_bytes,0.243281
1,num_src_ports,0.164578
10,dst_load,0.161456
13,dst_rate,0.150982
2,num_dst_ports,0.105752
